# Notebook 07: Evaluación Definitiva del Sistema Óptimo (Llama 3.3 + RAG Semántico)

## 1. Conclusiones de la Fase Comparativa
Tras la ejecución masiva y el análisis empírico de las 220 preguntas en el *Notebook 06*, los resultados han dictaminado la arquitectura óptima para nuestro sistema de Question Answering:

1. **El Motor de Razonamiento Ganador:** El modelo `Llama-3.3-70b-versatile` ha demostrado una capacidad de comprensión y síntesis notablemente superior a la del SLM local (Phi-3) y al grupo de control (Qwen 3), manejando la información con mayor rigor y precisión.
2. **El Motor de Recuperación Ganador:** El sistema **RAG Semántico (Vectorial basado en BGE-M3)** ha superado al RAG Léxico (BM25), logrando capturar el significado subyacente de las preguntas incluso cuando no compartían palabras clave exactas con los documentos de Elasticsearch.

## 2. Objetivo de este Cuaderno
El objetivo de este notebook es **ensamblar el sistema definitivo** uniendo a los dos ganadores: inyectaremos el contexto semántico recuperado por BGE-M3 directamente en Llama 3.3 70B. 

Este RAG Definitivo será sometido a la batería completa de 220 preguntas (Ground Truth). Las respuestas serán evaluadas bajo el mismo juez y el mismo *prompt* que en fases anteriores, garantizando la rigurosidad científica. Finalmente, se generará una visualización gráfica (Aciertos vs. Fallos) que representará el rendimiento final absoluto del Trabajo de Fin de Grado.


In [ ]:
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from dotenv import load_dotenv
from elasticsearch import Elasticsearch
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage
import torch

import warnings
warnings.filterwarnings("ignore")

# 1. Carga de API Key
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# 2. Conexión a Elasticsearch
print("Conectando a Elasticsearch...")
es = Elasticsearch("http://127.0.0.1:9250")

device = "cuda" if torch.cuda.is_available() else "cpu"

# 3. Carga del Modelo de Embeddings (BGE-M3)
print("Cargando BGE-M3 (Vectores)...")
embed_model = SentenceTransformer('BAAI/bge-m3', device=device)

# 4. Configuración de Modelos en Groq
llm_ganador = ChatGroq(
    model="llama-3.3-70b-versatile", 
    temperature=0, 
    api_key=GROQ_API_KEY
)

llm_juez = ChatGroq(
    model="openai/gpt-oss-120b", 
    temperature=0, 
    api_key=GROQ_API_KEY
)

# PROMPT DEL JUEZ
prompt_evaluacion = ChatPromptTemplate.from_messages([
    ("system", """Eres un Juez de Veracidad para un sistema RAG. 
    Tu única misión es validar si la 'Respuesta Generada' coincide con la 'Respuesta Esperada'.

    REGLAS DE ORO:
    1. La respuesta generada puede ser CORRECTA aunque este redactada de forma diferente, siempre que transmita el mismo dato o información esencial que la respuesta esperada.
    2. No uses tu propio conocimiento externo para validar la respuesta, ciñete a la comparación entre la respuesta esperada y la generada.

    Responde ÚNICAMENTE con la palabra "CORRECTO" o "INCORRECTO"."""),
    ("human", "Respuesta Esperada:\n{esperada}\n\nRespuesta Generada por el RAG:\n{generada}")
])

juez_chain = prompt_evaluacion | llm_juez | StrOutputParser()
print("Sistemas configurados correctamente.")

In [ ]:
def recuperar_contexto_definitivo(query, top_k=2):
    """Misma estructura de búsqueda vectorial que en rag_engine.py"""
    vector_query = embed_model.encode(query).tolist()
    
    search_payload = {
        "field": "vector_bge",
        "query_vector": vector_query,
        "k": top_k,
        "num_candidates": 100
    }
    
    res = es.knn_search(index="noticias_tfg", knn=search_payload, source=["title", "body"])
    
    contexto = ""
    for i, hit in enumerate(res['hits']['hits']):
        doc = hit['_source']
        contexto += f"\n--- NOTICIA {i+1} ---\nTITULO: {doc.get('title')}\nCUERPO: {doc.get('body')}\n"
    return contexto

def sistema_rag_ganador(pregunta):
    """Mismo prompt y estructura de mensajes que en rag_engine.py"""
    contexto = recuperar_contexto_definitivo(pregunta)
    
    prompt_completo = f"""
      Eres un sistema de verificación de datos (Fact-Checking). 
      Tu objetivo es responder a la pregunta usando ÚNICAMENTE el texto que te proporciono abajo.

      REGLAS CRÍTICAS:
      1. Si la respuesta no aparece en el texto o no hay informacion que responda a la pregunta, responde exactamente: "No tengo información suficiente en mis archivos".
      2. Formula tu respuesta usando solo los datos del texto. Puedes conectar ideas que estén en la noticia, pero NUNCA uses conocimiento externo ni inventes datos.
      3. No menciones otras noticias que no sean las proporcionadas.
      4. Las respuestas deberán ser cortas y directas.

      ### TEXTO DE REFERENCIA:
      {contexto}
      
      ### PREGUNTA:
      {pregunta}
    """
    
    messages = [HumanMessage(content=prompt_completo)]
    respuesta = llm_ganador.invoke(messages).content.strip()
    return respuesta

In [ ]:
# Cargamos la batería completa (220 preguntas)
df_preguntas = pd.read_csv("bateria_preguntas_tfg.csv", sep=";", encoding="utf-8-sig")

resultados_finales = []

print(f"Iniciando evaluación final sobre {len(df_preguntas)} preguntas...")

for index, row in tqdm(df_preguntas.iterrows(), total=len(df_preguntas)):
    pregunta = row['pregunta']
    esperada = row['respuesta_esperada']
    
    # 1. Generación de respuesta
    try:
        respuesta_rag = sistema_rag_ganador(pregunta)
        # 2. Evaluación con el Juez
        veredicto = juez_chain.invoke({"esperada": esperada, "generada": respuesta_rag}).strip().upper()
    except Exception as e:
        respuesta_rag = f"Error: {e}"
        veredicto = "INCORRECTO"

    # Limpieza de veredicto
    eval_final = "CORRECTO" if "CORRECTO" in veredicto else "INCORRECTO"
    
    resultados_finales.append({
        "Pregunta": pregunta,
        "Respuesta_Esperada": esperada,
        "RAG_Final_Resp": respuesta_rag,
        "Evaluacion_Juez": eval_final
    })
    
    time.sleep(0.5) # Límites de API

# Guardamos el resultado definitivo del TFG
df_res_final = pd.DataFrame(resultados_finales)
df_res_final.to_csv("resultados_RAG_Final_TFG.csv", sep=";", index=False, encoding="utf-8-sig")
print("Archivo 'resultados_RAG_Final_TFG.csv' generado con éxito.")

In [ ]:
# Cálculo de métricas
total = len(df_res_final)
aciertos = (df_res_final['Evaluacion_Juez'] == 'CORRECTO').sum()
fallos = total - aciertos
porcentaje = (aciertos / total) * 100

# Gráfico
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")
labels = ['Aciertos', 'Fallos']
counts = [aciertos, fallos]
colors = ['#4CAF50', '#F44336'] # Verde y Rojo académicos

ax = sns.barplot(x=labels, y=counts, palette=colors)

for p, val in zip(ax.patches, counts):
    ax.annotate(f"{val} ({ (val/total)*100:.1f}%)", 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', fontsize=12, color='black', xytext=(0, 5),
                textcoords='offset points')

plt.title(f'Rendimiento Final del Sistema RAG Ensamblado (Llama 3.3 + Vectorial)\nMuestra Total: {total} Preguntas', fontsize=14)
plt.ylabel('Número de Respuestas')
plt.ylim(0, total + 20)
plt.savefig("grafico_final_tfg.png", dpi=300)
plt.show()

print(f"Métrica Final de Precisión: {porcentaje:.2f}%")